# Connected-Component Structural Analysis — No Retraining

This notebook strengthens the final Warli DCGAN vs WGAN-GP study **without retraining**.

It:
- loads the existing epoch-100 checkpoints for seeds 42, 123, and 2024;
- evaluates the complete 998-image real dataset;
- generates a fixed pool of 500 images from each trained model;
- computes three foreground-structure measures:
  1. number of connected components,
  2. largest-component ratio,
  3. small-fragment count;
- reports architecture-level mean ± between-seed SD;
- exports CSV tables and publication-quality figures.

The foreground convention follows the manuscript's existing structure-aware analysis: pixels with normalized intensity **>= 0.55** are foreground.


In [ ]:
# Install only the packages needed for this analysis
!pip install -q scikit-image scipy pandas matplotlib


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from skimage.measure import label
from scipy import stats

# ------------------------------------------------------------
# Paths — these match the final multi-seed notebook
# ------------------------------------------------------------
DATA_DIR = Path("/content/drive/MyDrive/Warli_Project/Dataset/train")
RESULTS_DIR = Path("/content/drive/MyDrive/Warli_Project/MultiSeed/full_3seed")
OUT_DIR = RESULTS_DIR / "connected_component_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 2024]
IMAGE_SIZE = 64
LATENT_DIM = 100
THRESHOLD = 0.55
N_GENERATED = 500
SMALL_COMPONENT_MAX_PIXELS = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Dataset:", DATA_DIR)
print("Existing results:", RESULTS_DIR)
print("New outputs:", OUT_DIR)


In [ ]:
# ------------------------------------------------------------
# Generator architecture used in the final experiment
# ------------------------------------------------------------
class DCGANGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, feature_maps=64):
        super().__init__()
        fm = feature_maps
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, fm * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(fm * 8), nn.ReLU(True),
            nn.ConvTranspose2d(fm * 8, fm * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm * 4), nn.ReLU(True),
            nn.ConvTranspose2d(fm * 4, fm * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm * 2), nn.ReLU(True),
            nn.ConvTranspose2d(fm * 2, fm, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm), nn.ReLU(True),
            nn.ConvTranspose2d(fm, 1, 4, 2, 1, bias=False),
            nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z)

# Final WGAN-GP experiment uses the same generator topology.
class WGANGenerator(DCGANGenerator):
    pass


In [ ]:
# ------------------------------------------------------------
# Load the complete real dataset exactly at 64 x 64 grayscale
# ------------------------------------------------------------
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(str(DATA_DIR), transform=transform)

real_images = []
for i in range(len(dataset)):
    img, _ = dataset[i]
    real_images.append(img.squeeze(0).numpy())
real_images = np.stack(real_images)

print("Real images:", real_images.shape)
if len(real_images) != 998:
    print("WARNING: manuscript states 998 images; loaded", len(real_images))


In [ ]:
# ------------------------------------------------------------
# Connected-component metrics
# ------------------------------------------------------------
def connected_component_metrics(image01, threshold=THRESHOLD,
                                small_component_max_pixels=SMALL_COMPONENT_MAX_PIXELS):
    # Foreground convention consistent with the existing symmetry metric.
    mask = np.asarray(image01) >= threshold

    # 8-connectivity is appropriate for thin diagonal line structures.
    lab = label(mask, connectivity=2)
    component_ids, counts = np.unique(lab[lab > 0], return_counts=True)

    foreground_pixels = int(mask.sum())
    n_components = int(len(component_ids))

    if foreground_pixels == 0 or n_components == 0:
        return {
            "n_components": 0,
            "largest_component_ratio": 0.0,
            "small_fragment_count": 0,
            "foreground_fraction": 0.0,
        }

    largest_component_ratio = float(counts.max() / foreground_pixels)
    small_fragment_count = int(np.sum(counts <= small_component_max_pixels))

    return {
        "n_components": n_components,
        "largest_component_ratio": largest_component_ratio,
        "small_fragment_count": small_fragment_count,
        "foreground_fraction": float(foreground_pixels / mask.size),
    }

def analyse_image_set(images, source, seed=None):
    rows = []
    for idx, img in enumerate(images):
        m = connected_component_metrics(img)
        rows.append({
            "source": source,
            "seed": seed,
            "image_index": idx,
            **m
        })
    return pd.DataFrame(rows)

real_df = analyse_image_set(real_images, "Real", seed=np.nan)
real_df.to_csv(OUT_DIR / "real_connected_component_metrics.csv", index=False)
real_df.describe()


In [ ]:
# ------------------------------------------------------------
# Load existing epoch-100 checkpoints and generate fixed samples
# NO TRAINING occurs in this cell.
# ------------------------------------------------------------
def find_checkpoint(architecture, seed):
    if architecture == "DCGAN":
        candidates = [
            RESULTS_DIR / f"DCGAN_seed{seed}" / "checkpoints" / f"dcgan_seed{seed}_epoch_100.pth",
            RESULTS_DIR / f"DCGAN_seed{seed}" / "checkpoints" / f"dcgan_seed{seed}_epoch_100.pt",
        ]
    else:
        candidates = [
            RESULTS_DIR / f"WGAN_GP_seed{seed}" / "checkpoints" / f"wgan_gp_seed{seed}_epoch_100.pth",
            RESULTS_DIR / f"WGAN-GP_seed{seed}" / "checkpoints" / f"wgan_gp_seed{seed}_epoch_100.pth",
        ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Epoch-100 checkpoint not found for {architecture}, seed {seed}. "
        f"Checked: {candidates}"
    )

# One fixed latent pool is shared by all six trained models.
noise_gen = torch.Generator().manual_seed(2026)
fixed_noise = torch.randn(N_GENERATED, LATENT_DIM, 1, 1, generator=noise_gen)

generated_frames = []

for architecture in ["DCGAN", "WGAN-GP"]:
    model_class = DCGANGenerator if architecture == "DCGAN" else WGANGenerator

    for seed in SEEDS:
        ckpt_path = find_checkpoint(architecture, seed)
        print("Loading:", ckpt_path)

        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model = model_class().to(DEVICE)
        model.load_state_dict(checkpoint["generator_state_dict"])
        model.eval()

        batches = []
        with torch.no_grad():
            for start in range(0, N_GENERATED, 100):
                z = fixed_noise[start:start+100].to(DEVICE)
                x = ((model(z) + 1.0) / 2.0).clamp(0, 1)
                batches.append(x.cpu())

        generated = torch.cat(batches, dim=0).squeeze(1).numpy()
        df = analyse_image_set(generated, architecture, seed=seed)
        generated_frames.append(df)
        df.to_csv(
            OUT_DIR / f"{architecture.replace('-', '_')}_seed{seed}_connected_components.csv",
            index=False
        )

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

generated_df = pd.concat(generated_frames, ignore_index=True)
all_image_metrics = pd.concat([real_df, generated_df], ignore_index=True)
all_image_metrics.to_csv(OUT_DIR / "all_connected_component_image_metrics.csv", index=False)

print("Generated-image rows:", len(generated_df))
generated_df.head()


In [ ]:
# ------------------------------------------------------------
# Per-seed summaries and architecture-level mean ± between-seed SD
# ------------------------------------------------------------
metric_cols = [
    "n_components",
    "largest_component_ratio",
    "small_fragment_count",
    "foreground_fraction",
]

seed_summary = (
    generated_df
    .groupby(["source", "seed"])[metric_cols]
    .mean()
    .reset_index()
)
seed_summary.to_csv(OUT_DIR / "connected_component_per_seed_summary.csv", index=False)

architecture_rows = []
for architecture, sub in seed_summary.groupby("source"):
    row = {"Architecture": architecture, "N seeds": len(sub)}
    for metric in metric_cols:
        row[f"{metric} mean"] = sub[metric].mean()
        row[f"{metric} between-seed SD"] = sub[metric].std(ddof=1)
    architecture_rows.append(row)

architecture_summary = pd.DataFrame(architecture_rows)

real_summary = pd.DataFrame([{
    "Architecture": "Real",
    "N seeds": np.nan,
    **{f"{m} mean": real_df[m].mean() for m in metric_cols},
    **{f"{m} image-level SD": real_df[m].std(ddof=1) for m in metric_cols},
}])

architecture_summary.to_csv(
    OUT_DIR / "connected_component_architecture_summary.csv", index=False
)

print("=== Per-seed generated-image summaries ===")
display(seed_summary.round(4))

print("\n=== Architecture-level mean and between-seed SD ===")
display(architecture_summary.round(4))

print("\n=== Real-image reference distribution ===")
display(real_summary.round(4))


In [ ]:
# ------------------------------------------------------------
# Distance from the real structural reference
# Smaller absolute difference from the real mean = closer structural match.
# ------------------------------------------------------------
comparison_rows = []
for architecture, sub in seed_summary.groupby("source"):
    row = {"Architecture": architecture}
    for metric in ["n_components", "largest_component_ratio", "small_fragment_count"]:
        real_mean = real_df[metric].mean()
        arch_mean = sub[metric].mean()
        row[f"Real {metric}"] = real_mean
        row[f"Generated {metric}"] = arch_mean
        row[f"Absolute difference {metric}"] = abs(arch_mean - real_mean)
    comparison_rows.append(row)

real_distance = pd.DataFrame(comparison_rows)
real_distance.to_csv(OUT_DIR / "connected_component_distance_from_real.csv", index=False)
display(real_distance.round(4))


In [ ]:
# ------------------------------------------------------------
# Publication figure: distributions for the three principal metrics
# Each metric is saved as a separate figure.
# ------------------------------------------------------------
plot_order = ["Real", "DCGAN", "WGAN-GP"]
plot_data = {
    "Real": real_df,
    "DCGAN": generated_df[generated_df["source"] == "DCGAN"],
    "WGAN-GP": generated_df[generated_df["source"] == "WGAN-GP"],
}

labels = {
    "n_components": "Number of connected components",
    "largest_component_ratio": "Largest-component ratio",
    "small_fragment_count": "Small-fragment count",
}

for metric, ylabel in labels.items():
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    values = [plot_data[name][metric].to_numpy() for name in plot_order]
    ax.boxplot(values, labels=plot_order, showfliers=False)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Image source")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"CC_{metric}.png", dpi=600, bbox_inches="tight")
    fig.savefig(OUT_DIR / f"CC_{metric}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# ------------------------------------------------------------
# Optional exploratory paired seed-level comparisons.
# n=3, so treat these only as descriptive/exploratory.
# ------------------------------------------------------------
paired_rows = []
for metric in ["n_components", "largest_component_ratio", "small_fragment_count"]:
    d = seed_summary[seed_summary.source == "DCGAN"].sort_values("seed")[metric].to_numpy()
    w = seed_summary[seed_summary.source == "WGAN-GP"].sort_values("seed")[metric].to_numpy()

    t_stat, p_value = stats.ttest_rel(d, w)
    diff = d - w
    dz = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else np.nan

    paired_rows.append({
        "metric": metric,
        "n_pairs": len(diff),
        "DCGAN_mean": d.mean(),
        "WGAN_GP_mean": w.mean(),
        "mean_paired_difference_D_minus_W": diff.mean(),
        "paired_t": t_stat,
        "p_value_exploratory": p_value,
        "paired_Cohens_dz": dz,
    })

paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv(OUT_DIR / "connected_component_paired_exploratory.csv", index=False)
display(paired_df.round(4))

print("\nDONE. All outputs written to:")
print(OUT_DIR)
